In [ ]:
from sampo.scheduler.genetic import GeneticScheduler





# class MyGeneticScheduler(GeneticScheduler):
#     @staticmethod
#     def generate_first_population(...):
#         # твоя версия
#         ...

In [46]:
from sampo.schemas.graph import WorkGraph
from sampo.schemas.time import Time
from sampo.schemas.scheduled_work import ScheduledWork


from scripts.wg_converter import WorkGraphConverter, ProjectConverter
from sampo_api import contractor


wg = WorkGraph.loadf('wgs/small_synth', 'wg_11')
work_units = {node.id: node.work_unit for node in wg.nodes}


def read_file(path='wgs/small_synth', file='wg_11', contractors_N=5):  
    wg = WorkGraph.loadf(path, file)
    contractors = contractor(N = contractors_N)
    conv = WorkGraphConverter()
    data = conv.convert(wg, contractors)['rcpsp_data']
    return data

data = read_file()


In [47]:
ProjectConverter(wg, contractor(N = 5))

In [48]:
from sampo.utilities.validation import validate_schedule
from sampo.schemas.schedule_spec import ScheduleSpec

def validate_schedule_bool(schedule_obj, work_graph, contractors, spec = ScheduleSpec()):
    try:
        validate_schedule(schedule_obj, work_graph,  contractors, ScheduleSpec())
        return True
    except AssertionError:
        return False


In [49]:
import os
from scripts.valid import run_heuristic
from sampo.utilities.validation import validate_schedule



path = 'Heuristics/deepseek_chat'
files = [f for f in os.listdir(path) if f not in ('Steps', 'original_heuristics.json')]

pc = ProjectConverter(wg, contractor(N = 5))
for f in files:
    schedule, order, resource_usage, job_usage,  makespan = run_heuristic(path, f, data)
    schedule_obj =  pc.to_schedule(schedule, order, job_usage,  makespan)

    if validate_schedule_bool(schedule_obj, wg, contractor(N=5), ScheduleSpec()):
        print(f)
      
    


Longest Processing Time Priority Rule (LPT)
Minimum Slack Priority Rule (MSP)
Most Successors Priority Rule (MSP2)
Minimum Latest Finish Time Priority Rule (LFT2)
Shortest Processing Time Priority Rule (SPT)
Greatest Resource Demand Priority Rule (GRD)
Earliest Finish Time Priority Rule (EFT)
Earliest Start Time Priority Rule (EST)
Latest Finish Time Priority Rule (LFT)


# My Genetic Algrotihm 

In [73]:
# /Users/kimelfeld/Desktop/DSL and RCPSP/6_genetic.ipynb
from sampo.scheduler.genetic import GeneticScheduler
from scripts.wg_converter import WorkGraphConverter, ProjectConverter
import json
from scripts.valid import validate_schedule_bool, interpter_solver
from sampo.scheduler.genetic.schedule_builder import build_schedules, build_schedules_with_cache
from sampo.schemas.schedule import Schedule
from sampo.schemas.schedule_spec import ScheduleSpec
from sampo.schemas.landscape import LandscapeConfiguration



class LLMHeuristicsGeneticScheduler(GeneticScheduler):
    def __init__(self,
                 solvers_by,
                 mutate_order,
                 mutate_resources,
                 size_of_population,
                 number_of_generation = 50):
        self.solvers_by_path = os.path.join('Heuristics', solvers_by)
        self.llm_heuristics = self.get_llm_heuristics()
        super().__init__(number_of_generation=number_of_generation, 
                        mutate_order=mutate_order, 
                        mutate_resources=mutate_resources, 
                        size_of_population=size_of_population)
        
        #super().generate_first_population()


    def get_filter(self):
        with open(os.path.join(self.solvers_by_path, 'original_heuristics.json'), "r", encoding="utf-8") as f:
            filter = set(json.load(f))
        return filter
    
    def get_llm_heuristics(self):
        filter = self.get_filter()
        files = [file for file in os.listdir(self.solvers_by_path) if file not in ('Steps', 'original_heuristics.json')]
        heuristics = {}
        for file in files:
            if file.split(' ')[-1] not in filter:
                continue
            with open(os.path.join(self.solvers_by_path, file), "r", encoding="utf-8") as f:
                code = f.read()
            heuristics[file] = code
        return heuristics
    
    def set_converter(self, work_graph, contractors):
        ProjectConverter(work_graph, contractors)

    def to_schedule(self, work_graph):
        return ProjectConverter()
    def to_input_solver(self, work_graph, contractors):
        return WorkGraphConverter().convert(work_graph, contractors)['rcpsp_data']
    
    def generate_llm_population(self, work_graph, contractors, spec = ScheduleSpec()):
        data = self.to_input_solver(work_graph, contractors)
        project_converter = ProjectConverter(work_graph, contractors)
        population = {}
        for method, code in self.llm_heuristics.items():
            schedule, order, _, job_usage,  makespan = interpter_solver(method, code, data)
            schedule_obj = project_converter.to_schedule(schedule, order, job_usage, makespan)
            graph_nodes = project_converter.get_list_graph_nodes(schedule_obj)
            if validate_schedule_bool(schedule_obj, work_graph, contractors, spec):
                population[method] = (schedule_obj, graph_nodes, spec, 1)  # Schedule, list(GraphNode), Spec, weight
        return population
    
    def schedule_with_cache(self, work_graph, 
                            contractors, spec = ScheduleSpec(), validate = False, 
                            assigned_parent_time = Time(0), timeline = None, landscape = LandscapeConfiguration()):
        
        
        
        basic_init_schedules = super().generate_first_population(work_graph, contractors, 
                                                                 landscape, spec,
                                                                 self.work_estimator,
                                                                 self._deadline, self._weights)
        
        
        init_schedules = basic_init_schedules | self.generate_llm_population(work_graph, contractors)

        print(len(init_schedules))

        mutate_order, mutate_resources, mutate_zones, size_of_population = self.get_params(work_graph.vertex_count)
        deadline = None if self._optimize_resources else self._deadline

        schedules = build_schedules(wg,
                                    contractors,
                                    size_of_population,
                                    self.number_of_generation,
                                    mutate_order,
                                    mutate_resources,
                                    mutate_zones,
                                    init_schedules,
                                    self.rand,
                                    spec,
                                    self._weights,
                                    None,
                                    landscape,
                                    self.fitness_constructor,
                                    self.fitness_weights,
                                    self.work_estimator,
                                    self.sgs_type,
                                    assigned_parent_time,
                                    timeline,
                                    self._time_border,
                                    self._max_plateau_steps,
                                    self._optimize_resources,
                                    deadline,
                                    self._only_lft_initialization,
                                    self._is_multiobjective)
        schedules = [
            (Schedule.from_scheduled_works(scheduled_works.values(), wg), schedule_start_time, timeline, order_nodes)
            for scheduled_works, schedule_start_time, timeline, order_nodes in schedules]

        if validate:
            for schedule, *_ in schedules:
                validate_schedule(schedule, wg, contractors, spec)

        return schedules
        
    
        

#print(LLMHeuristicsGeneticScheduler('deepseek_reasoner', 0.5, 0.2, 20, 100))

In [74]:
algs = LLMHeuristicsGeneticScheduler('deepseek_reasoner', 0.05, 0.05, 50, 100)
#print(algs)
print(len(algs.get_llm_heuristics()))

5


In [75]:
GeneticScheduler(number_of_generation=100, mutate_order=0.05, mutate_resources=0.05, size_of_population=50).schedule(wg, contractor(N=5))

[SAMPO] [INFO] Toolbox initialization & first population took 8.893013000488281 ms


Genetic optimizing took 8.099079132080078 ms


[SAMPO] [INFO] First population evaluation took 247.56908416748047 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(55.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(55.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(55.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(55.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(55.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(55.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(53.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(53.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(53.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(50.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(50.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(50.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(50.0,) --
[SA

In [76]:
algs.schedule(wg, contractor(N=5))

[SAMPO] [INFO] Toolbox initialization & first population took 11.740922927856445 ms


12
Genetic optimizing took 11.451959609985352 ms


[SAMPO] [INFO] First population evaluation took 246.90914154052734 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(57.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(54.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(54.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(54.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(54.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(54.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(54.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(53.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(53.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(53.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(53.0,) --
[SA

In [16]:
algs.generate_llm_population(wg, contractor(N=5))

[(<sampo.schemas.schedule.Schedule at 0x168e05c90>,
  [dbf40c7e-262a-4585-8f7f-9d4bcd2285f3,
   b96ae541-b974-a8fb-33df-a672d842b1d7,
   14d0696d-1cd8-612e-21ec-620c8e538253,
   04a03bfe-884b-50ac-1cb3-0eb04d282944,
   0effed9e-49d2-3d71-0539-a60be39bb07f,
   4dbfa50d-0f9d-034d-b60f-3c599ec2d49e,
   31e84563-2707-2512-28b3-3dcf88fb8d10,
   ba91fd55-b7c3-98b3-7e94-054f947f3f91,
   2f71c3d2-772b-4a7b-3385-3a45f6fb9004,
   a73402f9-1c11-0153-ab92-c851a87f44e7,
   bd75ffc7-5c21-789f-9dc1-c9a715254db1,
   a36ae872-497b-ce5c-c9a8-10446df9a208,
   48dbd556-703e-e4e7-8100-55a2be734428,
   bd0b3ee0-dc22-cb5f-e710-486deaea75b4,
   6b4ecdde-9659-e81a-26e6-987d1cd11280,
   5608d34c-5bb7-6d9a-e2ea-197c0d543207,
   5d075210-8a60-a103-2835-892dedab0195,
   2c5d3c07-b29c-58d0-504f-97f81630a5f3,
   f2c82ff7-c10e-84e5-e1a4-9fcb8b1612bb,
   46fb7874-47fa-b82b-7761-765af95abc60,
   fc67e3f6-0389-9c10-7fab-44bf4ae4ec97,
   80882b2e-8baa-055a-2f71-3d38abd52045,
   1c7ec281-d472-f701-3b39-0be079da28fb,
   58

In [13]:
SOLVER_REGISTRY = {}
def load_solvers_from_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        code = f.read()

    ns = {}
    exec(code, ns)  # в ns появится rcpsp_solver из этого файла [web:41]
    solver = ns.get("rcpsp_solver")
    if solver is None:
        raise ValueError(f"No rcpsp_solver in {file_path}")

    name = os.path.basename(file_path)  # уникальный ключ — имя файла
    SOLVER_REGISTRY[name] = solver

load_solvers_from_file('Heuristics/deepseek_chat/Greatest Resource Demand Priority Rule (GRD)')


In [ ]:
cc = lambda n, m : 1.0 / (6 * m ** 2) * (-2 * n ** 3 + 3 * n ** 2 + (6 * m ** 2 - 1) * n)
SOLVER_REGISTRY['Greatest Resource Demand Priority Rule (GRD)'](data['jobs'], data['resources_detailed'], cc)

In [ ]:
print(LLMHeuristicsGeneticScheduler(0.5, 0.2, 20))